# Clustering Pipeline
This notebook trains KMeans, Agglomerative (Hierarchical), DBSCAN, and Gaussian Mixture Models on the provided dataset, evaluates with Silhouette Score, visualizes clusters (2D & optional 3D), compares models, interprets the best model's clusters with business-friendly names, and saves the final labeled dataset and artifacts.

In [1]:
## 1) Import Required Libraries

# Standard libraries
import os
import warnings
from pathlib import Path

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score


warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,6)


In [2]:
## 3) Load dataset


df = pd.read_csv('finalclusteringdataset.csv')
print('Loaded dataset shape:', df.shape)

df.head()

Loaded dataset shape: (100002, 5)


,trip_frequency,average_booking_value,destination_diversity,session_duration,search_behavior
0,-0.908427,-0.649795,0.779566,0.220535,-1.231300
1,-0.452339,NaN,0.779566,-0.212597,1.220781
2,-0.452339,-0.490367,-1.156358,-0.444734,-0.005260
3,0.003749,-0.391374,-1.156358,-0.969480,1.220781
4,-0.452339,-0.241383,0.779566,0.518060,-0.005260


In [3]:
## 4) Quick EDA & feature selection

# Quick checks
print('\nData types:')
print(df.dtypes)

print('\nMissing values per column:')
print(df.isna().sum())

print('\nNumeric summary:')
print(df.describe().T)

# Choose numeric features for clustering by default
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove obvious id/time columns if present
for drop_candidate in ['id','ID','Id','customer_id','index']:
    if drop_candidate in numeric_cols:
        numeric_cols.remove(drop_candidate)

print('\nNumeric columns used for clustering:', numeric_cols)
X_raw = df[numeric_cols].copy()



Data types:
trip_frequency           float64
average_booking_value    float64
destination_diversity    float64
session_duration         float64
search_behavior          float64
dtype: object

Missing values per column:
trip_frequency           5000
average_booking_value    4995
destination_diversity    5000
session_duration         5000
search_behavior             0
dtype: int64

Numeric summary:
                          count          mean       std       min       25%  \
trip_frequency          95002.0  3.111364e-17  1.000005 -2.276691 -0.908427   
average_booking_value   95007.0 -6.656173e-18  1.000005 -2.687204 -0.671569   
destination_diversity   95002.0  2.022386e-16  1.000005 -1.543543 -0.769173   
session_duration        95002.0  3.590035e-18  1.000005 -1.132090 -0.792740   
search_behavior        100002.0  1.350004e-16  1.000005 -1.231300 -1.231300   

                            50%       75%       max  
trip_frequency         0.003749  0.459837  2.512234  
average_booking_

In [4]:
## 6) Define clustering models


n_clusters = 4
models = {
    'KMeans': KMeans(n_clusters=n_clusters, random_state=42),
    'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
    'GMM': GaussianMixture(n_components=n_clusters, random_state=42)
}

results = {}
scores = {}


In [5]:
## 7-10) Fit models, predict labels, compute Silhouette Scores

from collections import defaultdict

for name, model in models.items():
    print(f'Fitting: {name}')
    try:
        if name == 'GMM':
            model.fit(X_scaled)
            labels = model.predict(X_scaled)
        elif name == 'KMeans':
            model.fit(X_scaled)
            labels = model.labels_
        else:
            # Agglomerative & DBSCAN have fit_predict
            labels = model.fit_predict(X_scaled)
            # save DBSCAN or Agglomerative by pickle via joblib

        # Count clusters (excluding DBSCAN noise -1)
        unique_labels = set(labels)
        n_effective_clusters = len([l for l in unique_labels if l != -1])
        print(f'Labels found: {sorted(unique_labels)} (effective clusters: {n_effective_clusters})')

        # Compute silhouette only if valid
        if n_effective_clusters > 1:
            score = silhouette_score(X_scaled, labels)
        else:
            score = np.nan

        results[name] = {
            'model': model,
            'labels': labels,
            'n_clusters': n_effective_clusters,
            'silhouette': score
        }
        scores[name] = score
        print(f'{name} silhouette: {score}\n')
    except Exception as e:
        print(f'Error fitting {name}:', e)
        results[name] = {'model': model, 'labels': None, 'n_clusters': 0, 'silhouette': np.nan}
        scores[name] = np.nan

# Summary
pd.DataFrame([{'model':k, 'silhouette':v} for k,v in scores.items()]).sort_values('silhouette', ascending=False)


Fitting: KMeans
Error fitting KMeans: name 'X_scaled' is not defined
Fitting: Agglomerative
Error fitting Agglomerative: name 'X_scaled' is not defined
Fitting: DBSCAN
Error fitting DBSCAN: name 'X_scaled' is not defined
Fitting: GMM
Error fitting GMM: name 'X_scaled' is not defined


,model,silhouette
0,KMeans,NaN
1,Agglomerative,NaN
2,DBSCAN,NaN
3,GMM,NaN


In [6]:
## 11) Visualize clusters (2D using PCA) and optional 3D


def plot_2d_pca(labels, title, filename=None, annotate_centers=None):
    plt.figure(figsize=(10,7))
    palette = sns.color_palette('tab10', n_colors=len(set(labels)))
    # Map -1 to gray for DBSCAN noise
    colors = [ 'lightgray' if l==-1 else palette[l % len(palette)] for l in labels]
    plt.scatter(X_pca2[:,0], X_pca2[:,1], c=colors, s=40, alpha=0.8)
    plt.title(title)
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')
    if annotate_centers is not None:
        for idx, c in enumerate(annotate_centers):
            plt.scatter(c[0], c[1], marker='X', s=200, edgecolor='k')
            plt.text(c[0], c[1], f'C{idx}', fontsize=12, weight='bold')
    if filename:
        plt.savefig(FIG_DIR / filename, bbox_inches='tight', dpi=150)
    plt.show()

# Plot for each model
for name, res in results.items():
    if res['labels'] is None:
        continue
    labels = res['labels']
    centers = None
    if name in ['KMeans']:
        km = res['model']
        # compute centers in PCA space
        centers = pca2.transform(km.cluster_centers_)
    elif name == 'GMM':
        # compute means in original feature space
        gmm = res['model']
        centers = pca2.transform(gmm.means_)
    plot_2d_pca(labels, f'{name} clusters (2D PCA) - silhouette: {res["silhouette"]}', filename=f'{name}_2d.png', annotate_centers=centers)

# Optional 3D interactive for best model (if >2 components)
best_model_name = max(scores, key=lambda k: (scores[k] if not pd.isna(scores[k]) else -999))
print('Best model by silhouette (preliminary):', best_model_name, 'score:', scores[best_model_name])
try:
    fig = px.scatter_3d(x=X_pca3[:,0], y=X_pca3[:,1], z=X_pca3[:,2], color=results[best_model_name]['labels'].astype(str), title=f'3D PCA - {best_model_name}')
    fig.write_html(str(FIG_DIR / f'{best_model_name}_3d.html'))
    fig.show()
except Exception as e:
    print('3D plot skipped:', e)


Best model by silhouette (preliminary): KMeans score: nan
3D plot skipped: name 'X_pca3' is not defined


In [7]:
## 12) Comparison table of Silhouette Scores

score_df = pd.DataFrame([{'model':k, 'silhouette':v} for k,v in scores.items()]).sort_values('silhouette', ascending=False).reset_index(drop=True)
score_df
score_df.to_csv('silhouette_scores.csv', index=False)


In [8]:
## 13) Analyze best model clusters and assign business names

# Determine best model robustly
valid_scores = {k:v for k,v in scores.items() if not pd.isna(v)}
if len(valid_scores)==0:
    raise RuntimeError('No valid silhouette scores found; cannot select best model automatically.')
best_model_name = max(valid_scores, key=lambda k: valid_scores[k])
best = results[best_model_name]
print('Best model:', best_model_name, 'silhouette:', valid_scores[best_model_name])

# Attach labels to original df
df['cluster_label'] = best['labels']

# Compute cluster-wise means on original features
cluster_summary = df.groupby('cluster_label')[numeric_cols].mean()
cluster_counts = df['cluster_label'].value_counts()
cluster_summary['count'] = cluster_counts

# Heuristic mapping using common feature keywords
spend_keys = [c for c in numeric_cols if any(k in c.lower() for k in ['spend','amount','total','price','fare','cost','revenue'])]
freq_keys = [c for c in numeric_cols if any(k in c.lower() for k in ['freq','frequency','visits','trips'])]
recency_keys = [c for c in numeric_cols if any(k in c.lower() for k in ['recency','last','days_since','days'])]

print('\nDetected spend keys:', spend_keys)
print('Detected freq keys:', freq_keys)
print('Detected recency keys:', recency_keys)

cluster_summary_display = cluster_summary.copy()
cluster_summary_display

# Assign names based on heuristics
cluster_names = {}
for label, row in cluster_summary.iterrows():
    score_features = {}
    # compute spend/freq/recency z-scores if available
    if spend_keys:
        score_features['spend'] = row[spend_keys].mean()
    if freq_keys:
        score_features['freq'] = row[freq_keys].mean()
    if recency_keys:
        score_features['recency'] = row[recency_keys].mean()

    # default name
    name = f'Cluster {int(label)}'
    if score_features:
        # simple rules
        spend = score_features.get('spend', np.nan)
        freq = score_features.get('freq', np.nan)
        recency = score_features.get('recency', np.nan)
        # compare to overall means
        overall = df[numeric_cols].mean()
        overall_spend = overall[spend_keys].mean() if spend_keys else np.nan
        overall_freq = overall[freq_keys].mean() if freq_keys else np.nan
        overall_recency = overall[recency_keys].mean() if recency_keys else np.nan

        if (not np.isnan(spend)) and (not np.isnan(freq)):
            if (spend >= overall_spend) and (freq >= overall_freq):
                name = 'Frequent High-Spender'
            elif (spend >= overall_spend) and (freq < overall_freq):
                name = 'Occasional Explorer'
            elif (spend < overall_spend) and (freq < overall_freq):
                name = 'Budget Traveller'
            elif (spend < overall_spend) and (freq >= overall_freq):
                name = 'Loyal Local Traveller'
        else:
            # fallback: use spend only
            if (not np.isnan(spend)):
                if spend >= overall_spend:
                    name = 'High-Spender'
                else:
                    name = 'Budget Traveller'
    cluster_names[label] = name

print('\nCluster name mapping:')
print(cluster_names)

# Add names to df
df['cluster_name'] = df['cluster_label'].map(cluster_names)

# Save cluster summary and final dataset
cluster_summary.to_csv('cluster_summary.csv')
df.to_csv(CSV_OUTPUT, index=False)


RuntimeError: No valid silhouette scores found; cannot select best model automatically.

In [ ]:
## 14) Save artifacts & quick summary



print('\nSilhouette scores:')
print(score_df)

# Show cluster summary head
try:
    display(pd.read_csv('cluster_summary.csv').head())
except Exception:
    print('No cluster_summary.csv found yet.')
